In [1]:
import os
import psycopg
import dotenv
import pandas as pd

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
dotenv.load_dotenv()
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}
assert all([var_value != "" for var_value in list(postgres_credentials.values())])

connection.update(postgres_credentials)
TABLE_NAME = "clean_flats_price"


with psycopg.connect(**connection) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]


df = pd.DataFrame(data, columns=columns)


Добавляем отобранные на предудыщем этапе признаки

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pandas as pd
from autofeat import AutoFeatRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing  import OneHotEncoder
from sklearn.preprocessing import RobustScaler
import mlflow
import numpy as np

CENTER_LAT = 55.7520
CENTER_LON = 37.6175

dotenv.load_dotenv()
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")

df = df[df['ceiling_height'] != 27]

def haversine_distance(lat, lon):
    R = 6371  

    lat = np.radians(lat)
    lon = np.radians(lon)
    center_lat = np.radians(CENTER_LAT)
    center_lon = np.radians(CENTER_LON)

    dlat = lat - center_lat
    dlon = lon - center_lon

    a = np.sin(dlat / 2) ** 2 + np.cos(center_lat)* np.cos(lat) * np.sin(dlon / 2) ** 2
    

    return 2 * R * np.arcsin(np.sqrt(a))

df_copy = df.copy()
df_copy["distance_to_center"] = haversine_distance(
    df["latitude"],
    df["longitude"],
)

X = df_copy.drop(columns = ['price', 'building_id', 'studio'])
y = df_copy['price']

cat_features = df.select_dtypes(['object']).columns.tolist()
cat_features.append("building_type_int")
train_size = 0.8
shuffle=True

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=train_size, random_state=42, shuffle=shuffle)

autofeat_cols = [
    "total_area",
    "living_area",
    "kitchen_area",
    "rooms",
    "floor",
    "floors_total",
    "flats_count",
    "ceiling_height",
]
transformations = ["1/", "log",  "sqrt"]

auto = AutoFeatRegressor(
    feateng_steps=2,
    transformations=transformations,
    max_gb=10,
    feateng_cols=autofeat_cols
)
X_train_fe = auto.fit_transform(X_train, y_train)
X_test_fe = auto.transform(X_test)

rob_feat = [
    "kitchen_area",
    "living_area",
    "total_area",
    "ceiling_height",
]

selected_features = [
    "kitchen_area",
    "total_area",
    "ceiling_height",
    "building_type_int_2.0",
    "building_type_int_4.0",
    "building_type_int_5.0",
    "building_type_int_6.0",
    "id",
    "build_year",
    "latitude",
    "longitude",
    "has_elevator",
    "distance_to_center",
    "floor*floors_total",
    "floors_total*total_area",
    "kitchen_area*log(rooms)",
    "log(rooms)/floors_total",
    "flats_count*floors_total",
    "log(total_area)/ceiling_height",
    "ceiling_height*log(total_area)"
]

preprocessor = ColumnTransformer(
    [
        ("robust_scaler", RobustScaler(quantile_range=(10, 90)), rob_feat),
        ("categorical", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), cat_features)
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform="pandas")


X_train_transformed = preprocessor.fit_transform(X_train_fe, y_train,)

X_test_transformed = preprocessor.transform(X_test_fe)
X_train = X_train_transformed[selected_features]
X_test = X_test_transformed[selected_features]

Для подбора гиперпараметров будем использовать библиотеку optuna. Сначала протеристируем sampler TPE

In [10]:
from xgboost import XGBRegressor
import optuna
from optuna.integration.mlflow import MLflowCallback


EXPERIMENT_NAME = "flats_price"
RUN_NAME="hypeparams"
STUDY_DB_NAME = 'sqlite:///local.study.db'
STUDY_NAME = 'TPE'
TRACKING_SERVER_HOST = "localhost"
TRACKING_SERVER_PORT = 5000
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0)
    }
    model = XGBRegressor(**params, objective="reg:squarederror",  eval_metric="rmse", random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    return rmse

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflc = MLflowCallback(tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
                           metric_name="rmse", create_experiment=False,
                           mlflow_kwargs={'experiment_id': experiment_id, 'tags': {'mlflow.parentRunId': run_id}})

study = optuna.create_study(direction='minimize', study_name=STUDY_NAME,
                                 storage=STUDY_DB_NAME, load_if_exists=True, sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=10, callbacks=[mlflc])
best_params = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params}")

/tmp/ipykernel_1772/3604417512.py:33: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
[I 2026-08-18 21:32:47,875] A new study created in RDB with name: TPE
[I 2026-08-18 21:32:59,278] Trial 0 finished with value: 2502804.70176238 and parameters: {'n_estimators': 349, 'max_depth': 12, 'learning_rate': 0.11384177128291817, 'min_child_weight': 9, 'subsample': 0.9183049482476509, 'colsample_bytree': 0.7164496789741365}. Best is trial 0 with value: 2502804.70176238.
[I 2026-08-18 21:33:05,330] Trial 1 finished with value: 2610985.2476636805 and parameters: {'n_estimators': 452, 'max_depth': 7, 'learning_rate': 0.023634970799338715, 'min_child_weight': 7, 'subsample': 0.9093116570307946, 'colsample_bytree': 0.7957710196949129}. Best is trial 0 with value: 2502804.70176238.
[I 2026-08-18 21:33:36,467] Trial 2 finished with value: 2

Number of finished trials: 10
Best params: {'n_estimators': 703, 'max_depth': 8, 'learning_rate': 0.07530690221299437, 'min_child_weight': 5, 'subsample': 0.8218610503606728, 'colsample_bytree': 0.9841707155447389}


In [16]:
study.best_value

2476446.749035992

In [14]:
fig = optuna.visualization.plot_param_importances(study)
fig.write_image("hp_assets/tpe_param_importances.png")
fig.show() 


Самые выжные параметры с отрывом можно видеть это learning_rate, n_estimators, max_depth

In [15]:
fig = optuna.visualization.plot_timeline(study)
fig.show() 

/tmp/ipykernel_1772/2489585031.py:1: ExperimentalWarning:

plot_timeline is experimental (supported from v3.2.0). The interface can change in the future.



Самая лучшая модель по качеству оказалась второй худшей моделей по производительности.

Теперь проверим семпер

In [18]:
RUN_NAME = 'RandomSampler'
STUDY_NAME = "RandomSampler"
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0)
    }
    model = XGBRegressor(**params, objective="reg:squarederror",  eval_metric="rmse", random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    return rmse

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflc = MLflowCallback(tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
                           metric_name="rmse", create_experiment=False,
                           mlflow_kwargs={'experiment_id': experiment_id, 'tags': {'mlflow.parentRunId': run_id}})
study_2 = optuna.create_study(direction='minimize', study_name=STUDY_NAME,
                                 storage=STUDY_DB_NAME, load_if_exists=True, sampler=optuna.samplers.RandomSampler(seed=42))
study_2.optimize(objective, n_trials=10, callbacks=[mlflc])
best_params = study.best_params

print(f"Best_value: {study_2.best_value}")
print(f"Best params: {best_params}")

/tmp/ipykernel_1772/293665242.py:24: ExperimentalWarning:

MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.

[I 2026-08-18 21:46:44,646] A new study created in RDB with name: RandomSampler
[I 2026-08-18 21:48:38,437] Trial 0 finished with value: 2542885.582418505 and parameters: {'n_estimators': 437, 'max_depth': 12, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481}. Best is trial 0 with value: 2542885.582418505.
[I 2026-08-18 21:49:14,824] Trial 1 finished with value: 2543628.446541905 and parameters: {'n_estimators': 152, 'max_depth': 11, 'learning_rate': 0.07725378389307355, 'min_child_weight': 8, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978}. Best is trial 0 with value: 2542885.582418505.
[I 2026-08-18 21:50:17,438] Trial 2 finished with value: 2693310.468912077 and parameters: {'n_estimators': 850, 'max_depth': 5, 'learning_rate

Best_value: 2490403.850418112
Best params: {'n_estimators': 703, 'max_depth': 8, 'learning_rate': 0.07530690221299437, 'min_child_weight': 5, 'subsample': 0.8218610503606728, 'colsample_bytree': 0.9841707155447389}


In [20]:
model = XGBRegressor(n_estimators=703, learning_rate=0.07530690221299437, max_depth=8, min_child_weight=8, subsample=0.8218610503606728,
        colsample_bytree=0.9841707155447389, objective="reg:squarederror", eval_metric="rmse", random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)

errors = pd.DataFrame({
    "y_true": y_test,
    "y_pred": pred
})

errors["error"] = errors["y_true"] - errors["y_pred"]
errors["abs_error"] = np.abs(errors["error"])
errors["squared_error"] = errors["error"] ** 2
errors.sort_values("squared_error", ascending=False).head(20)

,y_true,y_pred,error,abs_error,squared_error
94394,150000.0,36046756.0,-35896756.0,35896756.0,1.288577e+15
71359,70000.0,22498772.0,-22428772.0,22428772.0,5.030498e+14
116089,70000.0,22145504.0,-22075504.0,22075504.0,4.873279e+14
104377,35550000.0,13889893.0,21660107.0,21660107.0,4.691602e+14
120148,39200000.0,18777560.0,20422440.0,20422440.0,4.170761e+14
4103,40000000.0,19918910.0,20081090.0,20081090.0,4.032502e+14
42038,3200000.0,21913996.0,-18713996.0,18713996.0,3.502136e+14
89371,3550000.0,22074338.0,-18524338.0,18524338.0,3.431511e+14
113152,39950000.0,21610724.0,18339276.0,18339276.0,3.363290e+14
36663,35000000.0,17065924.0,17934076.0,17934076.0,3.216311e+14


In [21]:
worst = errors.nlargest(20, "squared_error")

# Добавляем исходные признаки квартир
worst_objects = df_copy.loc[worst.index].copy()

worst_objects["y_true"] = worst["y_true"]
worst_objects["y_pred"] = worst["y_pred"]
worst_objects["abs_error"] = worst["abs_error"]
worst_objects["squared_error"] = worst["squared_error"]

worst_objects

,id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,...,longitude,ceiling_height,flats_count,floors_total,has_elevator,distance_to_center,y_true,y_pred,abs_error,squared_error
94394,108011,17630,10,21.000000,71.000000,3,False,False,130.000000,150000.0,...,37.484936,3.00,245,19,True,11.365019,150000.0,36046756.0,35896756.0,1.288577e+15
71359,80619,8738,7,10.000000,42.000000,1,False,False,65.000000,70000.0,...,37.576366,2.80,311,9,True,2.849062,70000.0,22498772.0,22428772.0,5.030498e+14
116089,134345,1698,6,12.000000,43.000000,2,False,False,64.000000,70000.0,...,37.635395,3.00,161,13,True,4.044973,70000.0,22145504.0,22075504.0,4.873279e+14
104377,120713,2805,1,7.000000,40.000000,3,False,False,57.000000,35550000.0,...,37.564320,2.80,75,5,False,6.064321,35550000.0,13889893.0,21660107.0,4.691602e+14
120148,139307,24195,10,0.000000,0.000000,2,False,False,68.599998,39200000.0,...,37.564213,3.00,1630,44,True,3.935317,39200000.0,18777560.0,20422440.0,4.170761e+14
4103,4334,893,2,10.800000,80.000000,4,False,False,111.800003,40000000.0,...,37.470383,3.00,64,6,True,14.760670,40000000.0,19918910.0,20081090.0,4.032502e+14
42038,46050,663,4,8.000000,18.400000,3,False,False,85.000000,3200000.0,...,37.612156,3.00,80,5,False,3.963338,3200000.0,21913996.0,18713996.0,3.502136e+14
89371,102098,4209,6,11.000000,30.000000,2,False,False,54.000000,3550000.0,...,37.565971,3.00,130,8,True,3.667449,3550000.0,22074338.0,18524338.0,3.431511e+14
113152,131002,18799,13,18.799999,41.200001,2,False,False,79.400002,39950000.0,...,37.803234,3.00,147,18,True,12.662986,39950000.0,21610724.0,18339276.0,3.363290e+14
36663,40061,14053,20,13.000000,0.000000,3,False,False,80.000000,35000000.0,...,37.540874,2.64,252,22,True,17.581161,35000000.0,17065924.0,17934076.0,3.216311e+14
